In [ ]:
def find_perfect_match(T_b):
    covers = 5
    matches = []
    
    # Precompute all possible ts values as a set for O(1) lookup
    valid_ts = {3, 4, 5}
    
    for n in range(2, 12):
        denominator = n - 1
        for ti in range(5, 25):
            remaining = T_b - covers - (n * ti)
            if remaining <= 0:
                break  # Since ti increases, break early
            
            # Check if remaining/denominator is integer and in valid_ts
            if remaining % denominator == 0:
                ts = remaining // denominator
                if ts in valid_ts:
                    calc = (n * ti) + ((n - 1) * ts) + covers
                    matches.append((n, ti, ts, calc))
    
    # Print results
    for n, ti, ts, calc in matches:
        print(f"Perfect Match: n={n}, ti={ti}mm, ts={ts}mm, total = {calc}mm")
    
    return matches


find_perfect_match(73)

Perfect Match: n=3, ti=20mm, ts=4mm, total = 73mm
Perfect Match: n=4, ti=14mm, ts=4mm, total = 73mm
Perfect Match: n=6, ti=8mm, ts=4mm, total = 73mm
Perfect Match: n=8, ti=5mm, ts=4mm, total = 73mm


In [4]:
def calculate_shape_factor(a, b, ti, edge_cover=4):
    """
    Calculates the shape factor S for an internal layer 
    according to EN 1337-3.
    
    Parameters:
    a : float - Overall width of the bearing (mm)
    b : float - Overall length of the bearing (mm)
    ti: float - Thickness of an individual internal rubber layer (mm)
    edge_cover: float - Thickness of the elastomer side cover (default 4mm)
    """
    
    # 1. Calculate effective dimensions (subtracting cover from both sides)
    a_prime = a - (2 * edge_cover)
    b_prime = b - (2 * edge_cover)
    
    # 2. Calculate effective plan area (A')
    a_effective = a_prime * b_prime
    
    # 3. Calculate force-free perimeter (lp)
    lp = 2 * (a_prime + b_prime)
    
    # 4. Calculate Shape Factor (S)
    # Formula: S = (a' * b') / (2 * (a' + b') * ti)
    s = a_effective / (lp * ti)
    
    return round(s, 3)

# Example usage with your previous function output:
# Let's say your match was n=5, ti=10, ts=4, a=300, b=400
a_dim = 300
b_dim = 400
ti_val = 10

shape_factor = calculate_shape_factor(a_dim, b_dim, ti_val)
print(f"The Shape Factor (S) is: {shape_factor}")

The Shape Factor (S) is: 8.367


In [12]:
def find_matches_with_shape_factor(a, b, T_e):
    """
    Calculates bearing configurations and their corresponding 
    Shape Factor (S) based on EN 1337-3.
    """
    covers = 5        # Top + Bottom cover (2.5mm + 2.5mm)
    edge_cover = 4    # Fixed edge cover
    matches = []
    
    # Effective dimensions for A' and lp
    a_prime = a - (2 * edge_cover)
    b_prime = b - (2 * edge_cover)
    effective_area = a_prime * b_prime
    perimeter_prime = 2 * (a_prime + b_prime)
    
    valid_ts = {3, 4, 5}
    
    for n in range(2, 12):
        denominator = n - 1
        for ti in range(5, 25):
            remaining = T_e - covers - (n * ti)
            
            if remaining <= 0:
                break
            
            if remaining % denominator == 0:
                ts = remaining // denominator
                if ts in valid_ts:
                    # Calculate Shape Factor S
                    # S = A' / (lp * ti)
                    s_factor = effective_area / (perimeter_prime * ti)
                    
                    total_h = (n * ti) + (denominator * ts) + covers
                    matches.append({
                        "n": n,
                        "ti": ti,
                        "ts": ts,
                        "S": round(s_factor, 2),
                        "T_e": total_h
                    })
    
    # Display Results
    print(f"{'n':<3} | {'ti':<4} | {'ts':<4} | {'S Factor':<10} | {'T_e':<8}")
    print("-" * 40)
    for m in matches:
        print(f"{m['n']:<3} | {m['ti']:<4} | {m['ts']:<4} | {m['S']:<10} | {m['T_e']:<8}")
    
    return matches

# Example Run:
result = find_matches_with_shape_factor(a=560, b=380, T_e=73)

n   | ti   | ts   | S Factor   | T_e     
----------------------------------------
3   | 20   | 4    | 5.56       | 73      
4   | 14   | 4    | 7.94       | 73      
6   | 8    | 4    | 13.89      | 73      
8   | 5    | 4    | 22.22      | 73      


In [ ]:
def calculate_compressive_strain(G, S, Fz_d, a, b, vx_d, vy_d, edge_cover=4):
    """
    Calculates the design strain due to compressive load (epsilon_c,d)
    according to EN 1337-3 Clause 5.3.3.2.
    
    Parameters:
    G     : float - Shear modulus (MPa or N/mm2)
    S     : float - Shape factor of the internal layer
    Fz_d  : float - Design vertical load (N)
    a     : float - Overall width (mm)
    b     : float - Overall length (mm)
    vx_d  : float - Max horizontal displacement in direction a (mm)
    vy_d  : float - Max horizontal displacement in direction b (mm)
    edge_cover: float - Side elastomer cover (default 4mm)
    """
    
    # 1. Calculate effective dimensions of the reinforcing plate (A1)
    # Subtracting the edge cover from both sides
    a_prime = a - (2 * edge_cover)
    b_prime = b - (2 * edge_cover)
    A1 = a_prime * b_prime
    
    # 2. Calculate reduced effective plan area (Ar)
    # Formula (9): Ar = A1 * (1 - (vx_d / a_prime) - (vy_d / b_prime))
    # Note: Using a_prime and b_prime as the reference effective lengths
    ratio_x = vx_d / a_prime
    ratio_y = vy_d / b_prime
    
    Ar = A1 * (1 - ratio_x - ratio_y)
    
    # Validation: Ar cannot be zero or negative for the formula to work
    if Ar <= 0:
        return float('inf') # Or handle as a design failure
    
    # 3. Calculate design strain due to compressive loads (epsilon_c,d)
    # Formula (8): epsilon_c,d = (1.5 * Fz_d) / (G * Ar * S)
    epsilon_cd = (1.5 * Fz_d) / (G * Ar * S)
    
    return {
        "A1": round(A1, 2),
        "Ar": round(Ar, 2),
        "epsilon_cd": round(epsilon_cd, 4)
    }

# Example Usage:
result = calculate_compressive_strain(G=0.9, S=12.5, Fz_d=1500000, a=380, b=530, vx_d=5, vy_d=5)
print(f"Design Strain: {result['epsilon_cd']}")

Design Strain: 1.9286


In [1]:
import math

def find_bearing_configs(a, b, target_Tb, G, Fz_d, vx_d, vy_d, alpha_ad, alpha_bd, K_L=1.0):
    """
    Complete EN 1337-3 Design Check.
    Checks geometry, Shape Factor, and all Design Strains.
    """
    covers = 5        # 2.5mm top + 2.5mm bottom
    t_cover = 2.5
    edge_cover = 4
    matches = []
    
    # 1. Resultant horizontal displacement (vector sum)
    v_xyd = math.sqrt(vx_d**2 + vy_d**2)
    
    # 2. Effective dimensions for reinforcement
    a_prime = a - (2 * edge_cover)
    b_prime = b - (2 * edge_cover)
    A1 = a_prime * b_prime
    lp = 2 * (a_prime + b_prime)
    
    # 3. Reduced Area Ar (Formula 9)
    ratio_x = vx_d / a_prime
    ratio_y = vy_d / b_prime
    Ar = A1 * (1 - ratio_x - ratio_y)
    
    valid_ts = {3, 4, 5}
    
    for n in range(2, 12):
        denominator = n - 1
        for ti in range(5, 25):
            # Checking if this combination fits the target Total Height (T_b)
            remaining = target_Tb - covers - (n * ti)
            
            if remaining <= 0:
                break
            
            if remaining % denominator == 0:
                ts = remaining // denominator
                if ts in valid_ts:
                    # T_q: Total rubber thickness (for shear)
                    T_q_rubber = (n * ti) + covers
                    
                    # S: Shape Factor
                    s_factor = A1 / (lp * ti)
                    
                    # eps_cd: Compressive Strain (Formula 8)
                    eps_cd = (1.5 * Fz_d) / (G * Ar * s_factor)
                    
                    # eps_qd: Shear Strain (Formula 10)
                    eps_qd = v_xyd / T_q_rubber
                    
                    # eps_ad: Rotation Strain (Formula 11)
                    sum_ti3 = (n * (ti**3)) + (2 * (t_cover**3))
                    num_ad = ((a_prime**2 * alpha_ad) + (b_prime**2 * alpha_bd)) * ti
                    eps_ad = num_ad / (2 * sum_ti3)
                    
                    # eps_td: Total Design Strain
                    eps_td = K_L * (eps_cd + eps_qd + eps_ad)
                    
                    # Calculate actual total height to verify
                    actual_Tb = T_q_rubber + (denominator * ts)
                    
                    # EN 1337-3 Limits: eps_qd <= 1.0 and eps_td <= 7.0
                    is_valid = (eps_qd <= 1.00) and (eps_td <= 7.00)
                    
                    matches.append({
                        "n": n, "ti": ti, "ts": ts, "T_b": actual_Tb,
                        "S": round(s_factor, 2),
                        "eps_cd": round(eps_cd, 3),
                        "eps_qd": round(eps_qd, 3),
                        "eps_ad": round(eps_ad, 3),
                        "eps_td": round(eps_td, 3),
                        "status": "PASS" if is_valid else "FAIL"
                    })
    
    # --- Display Results ---
    header = f"{'n':<2} | {'ti':<2} | {'ts':<2} | {'T_b':<3} | {'S':<5} | {'e_cd':<5} | {'e_qd':<5} | {'e_ad':<5} | {'e_td':<5} | {'STATUS'}"
    print(header)
    print("-" * len(header))
    for m in matches:
        print(f"{m['n']:<2} | {m['ti']:<2} | {m['ts']:<2} | {m['T_b']:<3} | {m['S']:<5} | {m['eps_cd']:<5} | {m['eps_qd']:<5} | {m['eps_ad']:<5} | {m['eps_td']:<5} | {m['status']}")
    
    return matches

# Run with example data
results = find_bearing_configs(
    a=560, b=380, target_Tb=73, 
    G=0.9, Fz_d=1500000, 
    vx_d=30, vy_d=15, 
    alpha_ad=0.005, alpha_bd=0.002
)

n  | ti | ts | T_b | S     | e_cd  | e_qd  | e_ad  | e_td  | STATUS
-------------------------------------------------------------------
3  | 20 | 4  | 73  | 5.56  | 2.42  | 0.516 | 0.749 | 3.686 | PASS
4  | 14 | 4  | 73  | 7.94  | 1.694 | 0.55  | 1.145 | 3.389 | PASS
6  | 8  | 4  | 73  | 13.89 | 0.968 | 0.633 | 2.321 | 3.922 | PASS
8  | 5  | 4  | 73  | 22.22 | 0.605 | 0.745 | 4.364 | 5.715 | PASS


In [25]:
def check_rotational_limitation(a, b, n, ti, Fz_d, alpha_ad, alpha_bd, G, edge_cover=4, K_rd=3.0):
    """
    EN 1337-3 Clause 5.3.3.6: Rotational Limitation Condition.
    Calculates vertical deflection vc using Equation (20).
    
    Eb (Bulk Modulus) = 2000 MPa
    """
    # 1. Effective Dimensions
    a_prime = a - (2 * edge_cover)
    b_prime = b - (2 * edge_cover)
    A_prime = a_prime * b_prime
    lp = 2 * (a_prime + b_prime)
    
    # 2. Shape Factor (S)
    S = A_prime / (lp * ti)
    
    # 3. Parameters for Vertical Deflection (Equation 20)
    Eb_bulk = 2000  # Bulk modulus in MPa
    
    # Calculation for a single internal layer deflection:
    # (Fz * ti / A') * ( 1 / (5 * G * S^2) + 1 / Eb )
    term_geometry = 1 / (5 * G * (S**2))
    term_bulk = 1 / Eb_bulk
    
    v_c_single_layer = (Fz_d * ti / A_prime) * (term_geometry + term_bulk)
    
    # 4. Total Vertical Deflection (Sum of all layers)
    # n internal layers. (Note: Cover layers are often calculated similarly 
    # but using their specific thickness. Here we apply it to n layers).
    sum_vzd = n * v_c_single_layer
    
    # 5. Rotational Limitation Check (Equation 13)
    # sum_vzd - ( (a'*alpha_ad + b'*alpha_bd) / Krd ) >= 0
    rotation_term = (a_prime * alpha_ad + b_prime * alpha_bd) / K_rd
    limit_value = sum_vzd - rotation_term
    
    is_satisfied = limit_value >= 0
    
    return {
        "S": round(S, 2),
        "sum_vzd": round(sum_vzd, 4),
        "rotation_term": round(rotation_term, 4),
        "limit_value": round(limit_value, 4),
        "satisfied": is_satisfied
    }

# Example Usage with your parameters:
result = check_rotational_limitation(
   a=560, b=380, n=5, ti=12, Fz_d=1500000, 
   alpha_ad=0.005, alpha_bd=0.002, G=0.9
)

print(result)

{'S': 9.26, 'sum_vzd': 1.3551, 'rotation_term': 1.168, 'limit_value': 0.1871, 'satisfied': True}


In [28]:
import math

def check_stability(a, b, target_Tb, G, Fz_d, vx_d, vy_d, alpha_ad, alpha_bd, K_rd=3.0):
    """
    EN 1337-3: Rotational Limitation (Eq 13) and Buckling Stability (Eq 15).
    """
    covers = 5        
    edge_cover = 4
    Eb_bulk = 2000    
    matches = []
    
    # 1. Effective dimensions
    a_prime = a - (2 * edge_cover)
    b_prime = b - (2 * edge_cover)
    A1 = a_prime * b_prime
    lp = 2 * (a_prime + b_prime)
    
    # 2. Reduced Area Ar (Required for Buckling check)
    Ar = A1 * (1 - (vx_d / a_prime) - (vy_d / b_prime))
    
    # 3. Rotation term (Constant for the geometry)
    rotation_term = (a_prime * alpha_ad + b_prime * alpha_bd) / K_rd
    
    valid_ts = {3, 4, 5}
    
    for n in range(2, 12):
        denominator = n - 1
        for ti in range(5, 25):
            remaining = target_Tb - covers - (n * ti)
            
            if remaining <= 0 or remaining % denominator != 0:
                continue
                
            ts = remaining // denominator
            if ts in valid_ts:
                # Shape Factor S
                S = A1 / (lp * ti)
                Te = (n * ti) + covers # Total elastomer thickness
                
                # --- A. Rotational Limitation (Uplift) ---
                term_geometry = 1 / (5 * G * (S**2))
                term_bulk = 1 / Eb_bulk
                v_c_single = (Fz_d * ti / A1) * (term_geometry + term_bulk)
                sum_vzd = n * v_c_single
                uplift_limit = sum_vzd - rotation_term
                
                # --- B. Buckling Stability (Eq 15) ---
                actual_pressure = Fz_d / Ar
                limiting_pressure = (2 * a_prime * G * S) / (3 * Te)
                
                # Buckling Status
                buckling_pass = actual_pressure < limiting_pressure
                # Rotational Status
                uplift_pass = uplift_limit >= 0
                
                actual_Tb = (n * ti) + (denominator * ts) + covers
                
                matches.append({
                    "n": n, "ti": ti, "ts": ts, "T_b": actual_Tb,
                    "sum_vzd": round(sum_vzd, 2),
                    "uplift": round(uplift_limit, 2),
                    "p_act": round(actual_pressure, 2),
                    "p_lim": round(limiting_pressure, 2),
                    "status": "PASS" if (buckling_pass and uplift_pass) else "FAIL"
                })
    
    # --- Display Results ---
    header = f"{'n':<2} | {'ti':<2} | {'ts':<2} | {'T_b':<3} | {'sum_vzd':<7} | {'Uplift':<7} | {'p_act':<6} | {'p_lim':<6} | {'STATUS'}"
    print(header)
    print("-" * len(header))
    for m in matches:
        print(f"{m['n']:<2} | {m['ti']:<2} | {m['ts']:<2} | {m['T_b']:<3} | {m['sum_vzd']:<7} | {m['uplift']:<7} | {m['p_act']:<6} | {m['p_lim']:<6} | {m['status']}")
    
    return matches

# Run
results = check_stability(
    a=560, b=380, target_Tb=73, 
    G=0.9, Fz_d=1500000, 
    vx_d=30, vy_d=15,
    alpha_ad=0.005, alpha_bd=0.002
)

n  | ti | ts | T_b | sum_vzd | Uplift  | p_act  | p_lim  | STATUS
-----------------------------------------------------------------
3  | 20 | 4  | 73  | 3.37    | 2.21    | 8.07   | 28.31  | PASS
4  | 14 | 4  | 73  | 1.65    | 0.48    | 8.07   | 43.09  | PASS
6  | 8  | 4  | 73  | 0.58    | -0.59   | 8.07   | 86.8   | FAIL
8  | 5  | 4  | 73  | 0.28    | -0.89   | 8.07   | 163.56 | FAIL


In [ ]:
import math

def check_sliding(a, b, target_Tb, Fx_d, Fy_d, Fz_dmin, vx_d, vy_d, Kf=0.6):
    """
    EN 1337-3 Clause 5.3.3.6: Non-sliding condition.
    Kf = 0.6 for concrete, 0.2 for other surfaces.
    """
    covers = 5
    edge_cover = 4
    matches = []
    
    # 1. Resultant Horizontal Force
    Fxy_d = math.sqrt(Fx_d**2 + Fy_d**2)
    
    # 2. Effective dimensions
    a_prime = a - (2 * edge_cover)
    b_prime = b - (2 * edge_cover)
    A1 = a_prime * b_prime
    
    # 3. Reduced Area Ar (based on maximum displacements)
    Ar = A1 * (1 - (vx_d / a_prime) - (vy_d / b_prime))
    
    # 4. Average compressive stress from minimum load
    sigma_m = Fz_dmin / Ar
    
    # 5. Friction Coefficient
    mu_e = 0.1 + (1.5 * Kf / sigma_m)
    
    valid_ts = {3, 4, 5}
    
    for n in range(2, 12):
        denominator = n - 1
        for ti in range(5, 25):
            remaining = target_Tb - covers - (n * ti)
            if remaining <= 0 or remaining % denominator != 0:
                continue
                
            ts = remaining // denominator
            if ts in valid_ts:
                # Sliding resistance
                F_friction_max = mu_e * Fz_dmin
                
                # Checks: 
                # 1. Resultant force <= friction resistance
                # 2. Sigma_m >= 3 MPa (Permanent load condition)
                sliding_pass = Fxy_d <= F_friction_max
                permanent_stress_pass = sigma_m >= 3.0
                
                actual_Tb = (n * ti) + (denominator * ts) + covers
                
                matches.append({
                    "n": n, "ti": ti, "ts": ts, "T_b": actual_Tb,
                    "sigma_m": round(sigma_m, 2),
                    "mu_e": round(mu_e, 3),
                    "F_friction": round(F_friction_max / 1000, 2), # kN
                    "F_horizontal": round(Fxy_d / 1000, 2),        # kN
                    "status": "PASS" if (sliding_pass and permanent_stress_pass) else "FAIL"
                })

    # --- Display Results ---
    header = f"{'n':<2} | {'ti':<2} | {'ts':<2} | {'sigma_m':<7} | {'mu_e':<5} | {'F_fric(kN)':<10} | {'F_hor(kN)':<10} | {'STATUS'}"
    print(header)
    print("-" * len(header))
    for m in matches:
        print(f"{m['n']:<2} | {m['ti']:<2} | {m['ts']:<2} | {m['sigma_m']:<7} | {m['mu_e']:<5} | {m['F_friction']:<10} | {m['F_horizontal']:<10} | {m['status']}")
    
    return matches

# Example Run:
results = check_sliding(
    a=560, b=380, target_Tb=73, 
    Fx_d=50000, Fy_d=20000, Fz_dmin=1000000, 
    vx_d=30, vy_d=15, Kf=0.6
)
print(results)

n  | ti | ts | sigma_m | mu_e  | F_fric(kN) | F_hor(kN)  | STATUS
-----------------------------------------------------------------
3  | 20 | 4  | 5.38    | 0.267 | 267.31     | 53.85      | PASS
4  | 14 | 4  | 5.38    | 0.267 | 267.31     | 53.85      | PASS
6  | 8  | 4  | 5.38    | 0.267 | 267.31     | 53.85      | PASS
8  | 5  | 4  | 5.38    | 0.267 | 267.31     | 53.85      | PASS
None


In [38]:
def check_reinforcement(a, b, target_Tb, Fz_d, vx_d, vy_d, fy=235, Kh=1, gamma_m=1.00):
    """
    EN 1337-3 Clause 5.3.3.5: Reinforcing plate thickness check.
    
    Parameters:
    fy      : Yield stress of steel (default 235 MPa for S235)
    Kh      : 1.0 without holes, 2.0 with holes
    gamma_m : Partial safety factor (default 1.00)
    """
    covers = 5
    edge_cover = 4
    Kp = 1.3  # Stress correction factor per standard
    matches = []
    
    # 1. Effective dimensions
    a_prime = a - (2 * edge_cover)
    b_prime = b - (2 * edge_cover)
    A1 = a_prime * b_prime
    
    # 2. Reduced Area Ar (based on maximum design displacements)
    Ar = A1 * (1 - (vx_d / a_prime) - (vy_d / b_prime))
    
    valid_ts_options = {3, 4, 5}
    
    for n in range(2, 12):
        denominator = n - 1
        for ti in range(5, 25):
            remaining = target_Tb - covers - (n * ti)
            
            if remaining <= 0 or remaining % denominator != 0:
                continue
                
            ts_actual = remaining // denominator
            
            if ts_actual in valid_ts_options:
                # 3. Calculate minimum required thickness (ts_min)
                # Equation (12): ts_min = (Kp * Fz_d * (t1 + t2) * Kh * gamma_m) / (Ar * fy)
                # For internal plates, t1 = t2 = ti
                t_sum = ti + ti 
                
                ts_min_calc = (Kp * Fz_d * t_sum * Kh * gamma_m) / (Ar * fy)
                
                # Standard check: Must be at least 2mm AND >= calculated ts_min
                ts_min_final = max(2.0, ts_min_calc)
                
                is_safe = ts_actual >= ts_min_final
                
                actual_Tb = (n * ti) + (denominator * ts_actual) + covers
                
                matches.append({
                    "n": n, 
                    "ti": ti, 
                    "ts_provided": ts_actual,
                    "ts_required": round(ts_min_calc, 4),
                    "Ar": round(Ar, 2),
                    "status": "PASS" if is_safe else "FAIL"
                })

    # --- Display Results ---
    header = f"{'n':<2} | {'ti':<2} | {'ts_prov':<7} | {'ts_req':<7} | {'Ar(mm2)':<10} | {'STATUS'}"
    print(header)
    print("-" * len(header))
    for m in matches:
        print(f"{m['n']:<2} | {m['ti']:<2} | {m['ts_provided']:<7} | {m['ts_required']:<7} | {m['Ar']:<10} | {m['status']}")
    
    return matches

# Example Run:
results = check_reinforcement(
    a=560, b=380, target_Tb=73, 
    Fz_d=1500000, vx_d=30, vy_d=15, 
    fy=235, Kh=1
)

n  | ti | ts_prov | ts_req  | Ar(mm2)    | STATUS
-------------------------------------------------
3  | 20 | 4       | 1.7854  | 185904.0   | PASS
4  | 14 | 4       | 1.2498  | 185904.0   | PASS
6  | 8  | 4       | 0.7142  | 185904.0   | PASS
8  | 5  | 4       | 0.4464  | 185904.0   | PASS


In [40]:
import math

def calculate_structure_loads(a, b, target_Tb, G, Fz_d, vx_d, vy_d, alpha_ad, alpha_bd):
    """
    EN 1337-3 Clause 5.3.3.7: Forces, moments, and deformations exerted on the structure.
    """
    covers = 5
    edge_cover = 4
    matches = []
    
    # 1. Effective Dimensions
    a_prime = a - (2 * edge_cover)
    b_prime = b - (2 * edge_cover)
    A_total = a * b  # Total plan area for contact pressure
    
    # 2. Vectorial Resultant Displacement
    v_xy = math.sqrt(vx_d**2 + vy_d**2)
    
    # 3. Interpolation for Ks (Table 4)
    # Ratio b/a (using effective dimensions as per standard moment calc)
    ratio_ba = b_prime / a_prime
    
    def get_ks(ratio):
        table_4 = {
            0.5: 137, 0.75: 100, 1.0: 86.2, 1.2: 80.4, 1.25: 79.3, 1.3: 78.4,
            1.4: 76.7, 1.5: 75.3, 1.6: 74.1, 1.7: 73.1, 1.8: 72.2, 1.9: 71.5,
            2.0: 70.8, 2.5: 68.3, 10.0: 61.9, 1000: 60 # 1000 represents infinity
        }
        ratios = sorted(table_4.keys())
        if ratio <= ratios[0]: return table_4[ratios[0]]
        if ratio >= ratios[-1]: return table_4[ratios[-1]]
        
        for i in range(len(ratios) - 1):
            if ratios[i] <= ratio <= ratios[i+1]:
                r1, r2 = ratios[i], ratios[i+1]
                k1, k2 = table_4[r1], table_4[r2]
                return k1 + (k2 - k1) * (ratio - r1) / (r2 - r1)
        return 70.0 # Fallback

    ks_value = get_ks(ratio_ba)

    valid_ts = {3, 4, 5}
    
    for n in range(2, 12):
        denominator = n - 1
        for ti in range(5, 25):
            remaining = target_Tb - covers - (n * ti)
            if remaining <= 0 or remaining % denominator != 0:
                continue
                
            ts = remaining // denominator
            if ts in valid_ts:
                Te = (n * ti) + covers # Total thickness of elastomer in shear
                
                # --- A. Mean Pressure ---
                sigma_c = Fz_d / A_total
                
                # --- B. Resultant Horizontal Force Rxy (Eq 17) ---
                Rxy = (A_total * G * v_xy) / Te
                
                # --- C. Restoring Moment M (Eq 18) ---
                # We calculate moment for alpha_ad (rotation across width a)
                # Formula: M = G * (alpha * a'^5 * b') / (n * ti^3 * Ks)
                # Standard uses ti^3 per layer; here n * ti^3 covers the stack
                denom_m = n * (ti**3) * ks_value
                M_a = (G * alpha_ad * (a_prime**5) * b_prime) / denom_m
                M_b = (G * alpha_bd * (b_prime**5) * a_prime) / denom_m # For rotation in b
                
                actual_Tb = (n * ti) + (denominator * ts) + covers
                
                matches.append({
                    "n": n, "ti": ti, "ts": ts,
                    "pressure": round(sigma_c, 2),
                    "Rxy_kN": round(Rxy / 1000, 2),
                    "Ma_kNm": round(M_a / 1e6, 2),
                    "Mb_kNm": round(M_b / 1e6, 2),
                    "T_b": actual_Tb
                })

    # --- Display Results ---
    header = f"{'n':<2} | {'ti':<2} | {'ts':<2} | {'Press(MPa)':<10} | {'Rxy(kN)':<8} | {'Ma(kNm)':<8} | {'Mb(kNm)':<8}"
    print(header)
    print("-" * len(header))
    for m in matches:
        print(f"{m['n']:<2} | {m['ti']:<2} | {m['ts']:<2} | {m['pressure']:<10} | {m['Rxy_kN']:<8} | {m['Ma_kNm']:<8} | {m['Mb_kNm']:<8}")
    
    return matches

# Example Run:
results = calculate_structure_loads(
    a=560, b=380, target_Tb=73, G=0.9, Fz_d=1500000, 
    vx_d=30, vy_d=15, alpha_ad=0.005, alpha_bd=0.002
)

n  | ti | ts | Press(MPa) | Rxy(kN)  | Ma(kNm)  | Mb(kNm) 
----------------------------------------------------------
3  | 20 | 4  | 7.05       | 98.83    | 32.13    | 2.65    
4  | 14 | 4  | 7.05       | 105.31   | 70.25    | 5.8     
6  | 8  | 4  | 7.05       | 121.2    | 251.01   | 20.71   
8  | 5  | 4  | 7.05       | 142.75   | 771.1    | 63.62   
